# Dataset + DataLoader (from scratch)

A real `Dataset`/`DataLoader` implementation for actual training data -- contrast with `llms/iterators.ipynb`, which covered the iterable/iterator protocol and the map-style vs. streaming split conceptually, but deliberately stopped short of a full from-scratch build. This picks that back up for real: tokenizing/loading data, batching, and collating, as prep for the GPT-2-from-scratch training loop.

**Recap: how this maps to the iterator/generator protocol already covered:**
- `Dataset` (map-style) = `__len__` + `__getitem__` only, same protocol as `MapStyleTextDataset`. No `__iter__` needed -- `DataLoader` does the indexing. Stateless: each `__getitem__(i)` call is independent, no need to remember where you left off.
- `IterableDataset` = `__iter__` only -- PyTorch never calls `__next__` on the dataset directly. Implemented as a generator (`yield` inside `__iter__`), same protocol as `StreamingTextDataset`, rather than a full `__iter__`/`__next__` class (no need for the extra state/picklability that motivated class-based iterators earlier). `__iter__` should return a *fresh* generator each call (not `self`), so multiple `DataLoader` workers don't share state.
- `DataLoader` itself implements the iterator protocol at the top level -- `for batch in dataloader:` drives its own `__iter__`/`__next__`, which internally pulls from whichever dataset style (indexing for map-style, advancing the generator for iterable-style) to assemble each batch.
- What's actually new here vs. the hand-rolled versions: subclassing PyTorch's `Dataset`/`IterableDataset` base classes (mostly a marker so `DataLoader` knows which style it's dealing with), and needing `__getitem__`/the generator to yield something collatable into a batch (tensors, or numbers/strings the default `collate_fn` can stack) rather than arbitrary Python objects.


**What "collatable" actually means, and how to make something collatable:**

`default_collate` -- the function `DataLoader` automatically uses as its `collate_fn` when you don't supply your own -- has a built-in rule for combining a list of N per-sample outputs into one batch, based on their type:
- **Tensor** -> `torch.stack(list, dim=0)` -> shape `(N, *original_shape)`. Requires every sample's tensor to be the *exact same shape*.
- **int/float** -> wrapped into a 1-D tensor via `torch.tensor(list)`.
- **string** -> left as a plain list of strings (can't tensor-ify text).
- **tuple/list/dict of the above** -> collated field-by-field (e.g. `{"input_ids": t, "label": t}` collates `input_ids` across samples, `label` across samples, separately).
- **anything else** (a custom object, unrecognized type) -> no rule exists -> `TypeError`.

The most common way this breaks: **sequence length**. `__getitem__`/the generator only ever handles one sample at a time, so a single tensor (say, tokenized text of length 7) is never a problem by itself. The mismatch only shows up once `DataLoader` tries to combine *multiple* samples of *different sequence lengths* (e.g. 7 and 12) into one batch tensor -- `torch.stack` needs identical shapes, so stacking `(7,)` and `(12,)` together fails. It's specifically the sequence-length dimension that varies sample to sample (unlike, say, fixed-size images, which are already uniform shape).

Two ways to fix it:
1. **Pad inside `__getitem__`** to a fixed length, so every returned tensor is already the same shape *before* it ever reaches batching -- no sample needs to know about any other. Simple, but wastes compute on padding when most sequences are shorter than your fixed length.
2. **Write a custom `collate_fn`** and pass it to `DataLoader(..., collate_fn=my_fn)`. It receives the raw list of N un-padded samples, and you pad/stack them yourself -- typically with `torch.nn.utils.rnn.pad_sequence` (already in Group 4 of `tensor_basics.ipynb`), padding only to the *max length in that batch* rather than some global fixed size. This only works in `collate_fn` because that's the first point in the pipeline where all N samples' lengths are visible together. This is what real training pipelines usually do.


In [9]:
# TODO: Implement a map-style Dataset `SquareDataset`:
# - takes a list of numbers in its constructor
# - implements __len__ and __getitem__(i)
# - __getitem__(i) returns an (x, x**2) pair, both as torch.tensor
# - fixed shape (scalars) on purpose, so a plain DataLoader should just work -- no custom collate_fn needed
#
# Then wrap it in a real DataLoader(dataset, batch_size=...), loop over it, and print the batch shapes.
import torch
from torch.utils.data import Dataset, DataLoader

class SquareDataset(Dataset):
    def __init__(self, list_):
        self._list = torch.as_tensor(list_)
        self._length = len(self._list)
    
    def __len__(self):
        return self._length
    
    def __getitem__(self, index):
        if index >= self._length:
            raise IndexError
        return (self._list[index], self._list[index] **2)

square_dataset = SquareDataset([1., 2., 3., 4.])
square_dataloader = DataLoader(dataset=square_dataset, batch_size=2)

for batch_indx, sample in enumerate(square_dataloader):
    print(batch_indx, sample)

print(next(iter(square_dataloader))[0])

0 [tensor([1., 2.]), tensor([1., 4.])]
1 [tensor([3., 4.]), tensor([ 9., 16.])]
tensor([1., 2.])


**Does `__init__` always convert the input to a tensor?**

No -- depends on the data:
- Small, purely numeric data that fits in memory (like `SquareDataset` above) -- converting the whole thing to one tensor in `__init__` is common and convenient.
- Real-world data (images, text, big datasets) -- usually *not*. `__init__` just stores raw references (file paths, strings, a numpy array, pre-tokenized ints), and the actual tensor conversion (plus any augmentation/transform) happens lazily inside `__getitem__`, per sample. Reason: eagerly converting everything in `__init__` means loading the entire dataset into memory upfront, which defeats the point of `Dataset` for large data -- `__getitem__` is supposed to be where "produce one ready sample" work happens.


In [67]:
# TODO: Implement an IterableDataset `ChunkedStreamDataset`:
# - takes one long list/sequence of numbers in its constructor (think: a long stream of token ids)
# - implements __iter__ using a generator (yield) -- NOT __getitem__/__len__
# - __iter__ yields fixed-size, non-overlapping chunks of length chunk_len, as tensors
# - drop the last chunk if it doesn't evenly divide -- don't pad it

# Every chunk is the same length, so this should also work with a plain DataLoader -- no custom collate_fn needed.
# Then wrap it in a real DataLoader(dataset, batch_size=...), loop over it, and print the batch shapes.

import torch
from torch.utils.data import IterableDataset, DataLoader

class ChunkedStreamDataset(IterableDataset):
    def __init__(self, token_ids, chunk_len):
        self._token_ids = token_ids
        self._chunk_len = chunk_len
        self._len = len(self._token_ids)
    
    def __iter__(self):
        count = 0
        while(count + self._chunk_len <= self._len):
            a = torch.tensor(self._token_ids[count: count + self._chunk_len])
            count += self._chunk_len
            yield a 

import numpy as np
square_dataset = ChunkedStreamDataset(np.arange(30), 3)
#square_dataset = ChunkedStreamDataset(np.arange(3), 5)

square_dataloader = DataLoader(dataset=square_dataset, batch_size=2)
for batch_indx, sample in enumerate(square_dataloader):
    print(batch_indx, sample)

print(next(iter(square_dataloader))[0]) 

0 tensor([[0, 1, 2],
        [3, 4, 5]])
1 tensor([[ 6,  7,  8],
        [ 9, 10, 11]])
2 tensor([[12, 13, 14],
        [15, 16, 17]])
3 tensor([[18, 19, 20],
        [21, 22, 23]])
4 tensor([[24, 25, 26],
        [27, 28, 29]])
tensor([0, 1, 2])


In [ ]:
# TODO: Implement `SimpleDataLoader(dataset, batch_size=1, shuffle=False, collate_fn=None)`
# -- no multi-worker, but handles BOTH Dataset and IterableDataset, plus collate_fn, shuffling, and batching.
#
# - Detect dataset type: isinstance(dataset, IterableDataset) vs. map-style (has __len__/__getitem__).
# - shuffle: only valid for map-style. If shuffle=True on an IterableDataset, raise an error
#   (matches real PyTorch -- no indexing to shuffle).
# - collate_fn: if none given, default to the real torch.utils.data.default_collate.
# - __iter__ (the core of it), two branches:
#   * Map-style: build list(range(len(dataset))), shuffle it if requested, chunk into groups of
#     batch_size, for each chunk do [dataset[i] for i in chunk] and yield collate_fn(samples).
#   * Iterable-style: get it = iter(dataset), pull up to batch_size items at a time via repeated
#     next() (catch StopIteration to know the stream's done, same pattern as MyRange), yield
#     collate_fn(samples) for each batch, including a final partial one if it doesn't divide evenly.
#
# No num_workers, no pin_memory -- just those four pieces.
#
# Test it against both SquareDataset and ChunkedStreamDataset above, with and without shuffle,
# and confirm the batches look the same as the real DataLoader's output.

###### my naive implementation ###############
import torch
from torch.utils.data import IterableDataset, Dataset, DataLoader
from torch.utils.data import default_collate
import random

class SimpleDataLoader:
    def __init__(self, dataset, batch_size=1, shuffle=False, collate_fn=None):
        self._dataset = dataset
        self._batch_size = batch_size
        self._shuffle = shuffle
        self._indices = None
        self._collate_fn = default_collate if collate_fn is None else collate_fn
        
        ### this we can check here!!
        if isinstance(self._dataset, IterableDataset):
            if self._shuffle:
                raise ValueError("shuffle cannot be true for iterable datasets")
        elif isinstance(self._dataset, Dataset):
            self._indices = list(range(len(dataset)))
            if self._shuffle:
                random.shuffle(self._indices)
            self._indices = torch.as_tensor(self._indices)
            self._indices_batch = self._indices.split(self._batch_size)
        else:
            raise TypeError("only dataset and IterableDataset accepted!!")
        
    
    def __iter__(self):
        
        if isinstance(self._dataset, IterableDataset):
            iter_dataset = iter(self._dataset)
            done = False
            while not done:
                samples = []
                for _ in range(self._batch_size):
                    try:
                        samples.append(next(iter_dataset))
                    except StopIteration:
                        done = True
                        break
                if samples:
                    yield self._collate_fn(samples)
                
        elif isinstance(self._dataset, Dataset):
            for current_indices in self._indices_batch:
                samples = [self._dataset[i] for i in current_indices]
                yield self._collate_fn(samples)

#square_dataset = SquareDataset([1., 2., 3., 4.])
square_dataset = ChunkedStreamDataset(np.arange(30), 3)

square_dataloader = SimpleDataLoader(dataset=square_dataset, batch_size=2, shuffle=False)
for batch_indx, sample in enumerate(square_dataloader):
    print(batch_indx, sample)

0 tensor([[0, 1, 2],
        [3, 4, 5]])
1 tensor([[ 6,  7,  8],
        [ 9, 10, 11]])
2 tensor([[12, 13, 14],
        [15, 16, 17]])
3 tensor([[18, 19, 20],
        [21, 22, 23]])
4 tensor([[24, 25, 26],
        [27, 28, 29]])
